# BioPortal Communication Term Search (No Definition Check)

This notebook provides a lightweight utility for searching terms in BioPortal ontologies using the BioPortal REST API. It supports **exact term matching** and **synonym-based lookup**, returning the best-matched ontology concept IRI along with the match type.

## Features

- Search ontology terms using the BioPortal `/search` endpoint  
- Detect and classify:
  - **Exact matches**
  - **Synonym matches**
- Return the mapped concept IRI for downstream processing  
- Interface for single-term lookups

## Requirements

The notebook expects:

- A valid **BioPortal API key** stored in `google.colab.userdata` as `bioportal-key`
- An active internet connection for API requests

## Main Function

### `find_term_in_ontology(term: str, ontology_abbrev: str) -> Tuple[str, str]`

Parameters:

- `term` — the search term
- `ontology_abbrev` — BioPortal ontology abbreviation (e.g., `"IOBC"`, `"GO"`, `"SNOMEDCT"`)

Returns:

- **IRI** of the matched ontology concept  
- **Match type** (`"exact"` or `"synonym"`)



In [6]:
# @title Imports and general specifications
from typing import List, Dict, Any, Tuple
from google.colab import userdata
import requests

bioportal_api = userdata.get('bioportal-key')
BASE_URL = "https://data.bioontology.org"


In [10]:
# @title Code implementation


def find_term_in_ontology(
    term: str,
    ontology: str,
    exact: bool = True,
    case_sensitive: bool = False
) -> Tuple[str, str]:
    """
    Function to search for a given term in a specified ontology without retrieving
    the definition.

    Searches a specified ontology in the BioPortal API for a given term and returns:
      - the mapped ID
      - the mapping type: "exact" or "synonym"

    If no matches are found and exact=True, the search is retried with exact=False.

    Parameters
    ----------
    term : str
        Search term.
    ontology : str
        Ontology acronym.
    exact : bool, optional
        If True, attempt exact prefLabel match first.
    case_sensitive : bool, optional
        If True, case-sensitive matching is used and no .lower() transformations occur.

    Returns
    -------
    Tuple[str, str]
        (mapped_id, match_type)
        Returns ("", "") if no match found.
    """

    params = {
        "q": term,
        "ontologies": ontology,
        "require_exact_match": str(exact).lower(),
        "include": "prefLabel,definition,synonym,notation,cui,semanticType",
        "pagesize": 20,
        "apikey": bioportal_api
    }

    resp = requests.get(f"{BASE_URL}/search", params=params, timeout=15)
    resp.raise_for_status()
    entries = resp.json().get("collection", [])

    # Apply case sensitivity rule
    if case_sensitive:
        term_cmp = term               # keep original case
    else:
        term_cmp = term.lower()       # lowercase for case-insensitive match

    # --- Filter logic ---
    filtered = []

    if exact:
        for e in entries:
            label = e.get("prefLabel", "")
            label_cmp = label if case_sensitive else label.lower()
            if label_cmp == term_cmp:
                filtered.append(e)
        match_type = "exact"
    else:
        for e in entries:
            syns = e.get("synonym") or []
            if isinstance(syns, str):
                syns = [syns]

            for s in syns:
                syn_cmp = s if case_sensitive else s.lower()
                if syn_cmp == term_cmp:
                    filtered.append(e)
                    break
        match_type = "synonym"

    # If nothing found and this was an exact search → retry as synonym search
    if not filtered and exact:
        return find_term_in_ontology(
            term,
            ontology,
            exact=False,
            case_sensitive=case_sensitive
        )

    # Still nothing found
    if not filtered:
        return "", ""

    # Extract mapped ID
    first = filtered[0]
    mapped_id = first.get("@id", first.get("id", ""))

    return mapped_id, match_type

In [11]:
# @title Check with a single term
# User specifies the term and the ontology abbreviation as in Bioportal.
# Note not all resulted IRIs are clicable URLS!
find_term_in_ontology('Inactivation','IOBC')

('http://purl.jp/bio/4/id/200906048346196865', 'synonym')

**Explanation**

Represented code made a search through the IOBC ontology and found the best fitting term. The search was based on the string similarity, with no semantic definition check.

You can verify the result for the term `Inactivation` in `IOBC` ontology directly in the bioportal.

See the following term link
https://bioportal.bioontology.org/ontologies/IOBC/?p=classes&conceptid=http%3A%2F%2Fpurl.jp%2Fbio%2F4%2Fid%2F200906048346196865
